# Notebook 6 – Outlier Treatment

Outlier Treatment is an important step in Data Preprocessing. It helps identify unusually high or low values in a dataset and decide how to handle them before performing Data Analysis or building Machine Learning models.

## What is an Outlier?
Definition
- An outlier is a data point that is unusually different from the majority of observations in a dataset.
- It may be significantly higher or lower than the typical values.
- Outliers can be identified using statistical methods, visualization, or domain knowledge.
#### Real-world example
Consider the following employee salaries:
20,000
25,000
30,000
35,000
40,000
5,00,000
Most salaries are between ₹20,000 and ₹40,000.
The salary ₹5,00,000 is unusually high compared with the other observations.
It may be an outlier, but it could represent a senior employee's genuine salary.
#### Business example
In an e-commerce dataset, most orders are worth ₹500–₹5,000.
One order worth ₹2,00,000 is unusually large.

It may represent:
- A bulk purchase.
- A corporate order.
- An incorrect amount.
- A fraudulent transaction.
#### AI/ML use case
Outliers can affect:
- Linear Regression.
- K-Means Clustering.
- Distance-based algorithms.
- Mean and standard deviation.
- Feature scaling.
- Statistical analysis.
For example, an extreme income value can distort customer segmentation using K-Means.

In [1]:
%mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 4.843 seconds
  Name    Version  Build                Channel
-------------------------------------------------------------
+ pandas  3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
- pip     26.2.2   pyh145f28c_0         conda-forge


In [2]:
import pandas as pd

df = pd.DataFrame({
    "Salary": [20000, 25000, 30000, 35000, 40000, 500000]
})

print(df.describe())

              Salary
count       6.000000
mean   108333.333333
std    192006.944319
min     20000.000000
25%     26250.000000
50%     32500.000000
75%     38750.000000
max    500000.000000


#### Explanation
- describe() provides statistical summaries.
- Mean, standard deviation, minimum, and maximum help understand the distribution.
- The maximum salary is much higher than most observations.
- This suggests a possible outlier that requires investigation.

## Outlier vs Error
Definition
- An outlier is an observation that is unusually different from other observations.
- An error is an incorrect value caused by data entry, measurement, processing, or other problems.
- An outlier may be valid, whereas an error represents incorrect data.
#### Real-world example
Age
22
25
28
150

Age 150 is an outlier and may also be an error.

However, the value 75 may be an outlier in a young employee dataset but still be valid in a general population dataset.
#### Business example
A transaction worth ₹10,00,000 is unusually large.

It could be:
- A valid bulk purchase.
- A duplicate transaction.
- An incorrect decimal placement.
- A fraudulent transaction.
#### AI/ML use case
The distinction is important because:
- Errors should generally be corrected or removed.
- Valid extreme observations may need to be retained.
- Some outliers require capping or transformation.
- Fraud-related outliers may be the most important observations in the dataset.

In [3]:
import pandas as pd

df = pd.DataFrame({
    "Age": [22, 25, 28, 30, 150]
})

# Identify values outside an example valid age range
invalid_age = (df["Age"] < 0) | (df["Age"] > 120)

print(df[invalid_age])

   Age
4  150


#### Explanation
- We defined an example valid age range of 0–120.
- Age 150 violates the defined rule.
- This is not merely a statistical outlier; it is also an invalid value under this rule.
- It should be investigated and corrected if possible.
Key difference: Statistical methods identify unusual values. Business rules help determine whether those values are incorrect.

## IQR Method (Interquartile Range)
Definition
- IQR is the difference between the third quartile (Q3) and the first quartile (Q1).
- It measures the spread of the middle 50% of the data.
- The IQR method identifies outliers using lower and upper statistical boundaries.

#### Business example
A retail company analyzes daily sales.

Most stores generate sales between ₹10,000 and ₹50,000, while one store records ₹5,00,000.

The IQR method can flag this store for investigation.
#### AI/ML use case
IQR is useful for:
- Salary prediction datasets.
- Customer spending datasets.
- House price analysis.
- Numerical feature preprocessing.
- Skewed datasets where normality is not assumed.

In [4]:
import pandas as pd

df = pd.DataFrame({
    "Salary": [
        20000, 25000, 30000, 35000,
        40000, 45000, 50000, 500000
    ]
})

# Calculate quartiles
Q1 = df["Salary"].quantile(0.25)
Q3 = df["Salary"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate boundaries
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Identify outliers
outliers = df[
    (df["Salary"] < lower) |
    (df["Salary"] > upper)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower)
print("Upper Bound:", upper)

print("\nOutliers:")
print(outliers)

Q1: 28750.0
Q3: 46250.0
IQR: 17500.0
Lower Bound: 2500.0
Upper Bound: 72500.0

Outliers:
   Salary
7  500000


#### Explanation
- quantile(0.25) calculates Q1.
- quantile(0.75) calculates Q3.
- IQR measures the spread of the middle 50%.
- The lower and upper boundaries define the potential outlier range.
- Values outside the boundaries are identified.
#### When to use IQR
- When the data is skewed.
- When extreme values may affect mean and standard deviation.
- When a robust statistical method is preferred.
- When you want a simple method for detecting potential outliers.
#### When not to rely on IQR alone
- When the dataset is very small.
- When extreme values are expected naturally.
- When the distribution has multiple groups.
- When the business context makes certain extreme observations important.

## Z-Score Method
Definition
- Z-Score measures how far a data point is from the mean in terms of standard deviations.
- It indicates whether a value is unusually high or low compared with the average.
- A common rule flags values with absolute Z-Score greater than 3 as potential outliers.

#### Business example
A manufacturing company measures the weight of products.
Most products weigh around 500 grams, but one product weighs 900 grams.
The Z-Score can help identify unusual measurements.
#### AI/ML use case
Z-Score is useful for:
- Detecting unusual numerical observations.
- Identifying anomalies in approximately normal distributions.
- Detecting extreme values in measurement datasets.
- Standardizing numerical features.

In [5]:
import pandas as pd

df = pd.DataFrame({
    "Salary": [
        30000, 35000, 40000,
        45000, 50000, 500000
    ]
})

# Calculate mean and standard deviation
mean = df["Salary"].mean()
std = df["Salary"].std()

# Calculate Z-Score
df["Z_Score"] = (
    (df["Salary"] - mean) / std
)

# Identify potential outliers
df["Outlier"] = df["Z_Score"].abs() > 3

print(df)

   Salary   Z_Score  Outlier
0   30000 -0.461171    False
1   35000 -0.434565    False
2   40000 -0.407959    False
3   45000 -0.381353    False
4   50000 -0.354747    False
5  500000  2.039796    False


#### Explanation
- mean() calculates the average salary.
- std() calculates the standard deviation.
- The Z-Score measures how far each salary is from the average.
- abs() > 3 identifies observations more than three standard deviations away.
#### When to use Z-Score
- When data is approximately normally distributed.
- When mean and standard deviation provide meaningful summaries.
- When you want to measure the distance of observations from the mean.
#### When not to rely on Z-Score alone
- When data is highly skewed.
- When the dataset contains extreme values that distort the mean and standard deviation.
- When the dataset has multiple populations or groups.
- When rare but valid observations are expected.

## Percentile Method
Definition
- The Percentile Method identifies extreme observations using specified percentiles.
- Percentiles indicate the position of a value within a distribution.
- Values below a lower percentile or above an upper percentile can be flagged as potential outliers.
#### Real-world example
A company analyzes customer spending.
It wants to identify customers in the lowest 1% and highest 1% of spending.
The 1st and 99th percentiles can be used as boundaries.
#### Business example
A company wants to investigate the highest-value 5% of orders without assuming that these orders are incorrect.
#### AI/ML use case
The percentile method is useful for:
- Income datasets.
- Customer spending.
- House prices.
- Highly skewed numerical features.
- Capping extreme values.

In [6]:
import pandas as pd

df = pd.DataFrame({
    "Income": [
        20000, 25000, 30000, 35000,
        40000, 45000, 50000, 60000,
        70000, 1000000
    ]
})

# Calculate percentile boundaries
lower = df["Income"].quantile(0.01)
upper = df["Income"].quantile(0.99)

# Identify potential outliers
outliers = df[
    (df["Income"] < lower) |
    (df["Income"] > upper)
]

print("Lower Percentile:", lower)
print("Upper Percentile:", upper)

print("\nPotential Outliers:")
print(outliers)

Lower Percentile: 20450.0
Upper Percentile: 916300.0000000001

Potential Outliers:
    Income
0    20000
9  1000000


#### Explanation
- quantile(0.01) calculates the 1st percentile.
- quantile(0.99) calculates the 99th percentile.
- Values outside these boundaries are flagged.
- The method is based on the chosen percentile thresholds.
#### When to use
- When you want to control how much of the data is flagged.
- When the dataset is skewed.
- When business requirements specify percentile boundaries.

## Winsorization
Definition
- Winsorization is a statistical technique that limits extreme values to specified percentile boundaries.
- Instead of deleting observations, it replaces extreme values with boundary values.
- It preserves the number of observations while reducing the influence of extreme values.

#### Business example
A retail company wants to analyze average customer spending without allowing a few extremely large purchases to dominate the analysis.
#### AI/ML use case
Winsorization is useful for:
- Linear Regression.
- Numerical feature preprocessing.
- Financial analysis.
- Customer spending analysis.
- Reducing the influence of extreme observations.

In [7]:
import pandas as pd
from scipy.stats.mstats import winsorize

df = pd.DataFrame({
    "Income": [
        20000, 25000, 30000,
        35000, 40000, 500000
    ]
})

# Winsorize 10% from each tail
df["Income_Winsorized"] = winsorize(
    df["Income"],
    limits=(0.10, 0.10)
)

print(df)

   Income  Income_Winsorized
0   20000              20000
1   25000              25000
2   30000              30000
3   35000              35000
4   40000              40000
5  500000             500000


#### Explanation
- winsorize() limits extreme values based on the selected tail proportions.
- limits=(0.10, 0.10) specifies 10% at the lower tail and 10% at the upper tail.
- Extreme values are replaced with boundary values rather than deleted.
- The total number of rows remains unchanged.

## Capping
Definition
- Capping replaces values above or below predefined boundaries with those boundary values.
- It reduces the effect of extreme observations without deleting records.
- Upper capping limits unusually high values, while lower capping limits unusually low values.
#### Real-world example
Suppose the allowed salary cap for an analysis is ₹1,00,000.

A salary of ₹5,00,000 is replaced with ₹1,00,000 for that analysis.
#### Business example
A company caps customer spending values at the 99th percentile to prevent a few large purchases from dominating customer segmentation.
#### AI/ML use case
Capping can help:
- Reduce the influence of extreme numerical features.
- Stabilize some regression models.
- Improve numerical feature distributions.
- Preserve observations while limiting their magnitude.

In [8]:
import pandas as pd

df = pd.DataFrame({
    "Income": [20000, 30000, 40000, 50000, 500000]
})

# Define boundaries
lower = 20000
upper = 100000

# Apply capping
df["Income_Capped"] = df["Income"].clip(
    lower=lower,
    upper=upper
)

print(df)

   Income  Income_Capped
0   20000          20000
1   30000          30000
2   40000          40000
3   50000          50000
4  500000         100000


#### Explanation
- clip() limits values to the specified range.
- Values below ₹20,000 become ₹20,000.
- Values above ₹1,00,000 become ₹1,00,000.
- Values within the range remain unchanged.
#### When to use capping
- When extreme values are valid but excessively influential.
- When preserving the number of observations is important.
- When business rules or statistical boundaries justify limiting values.

## Flooring
Definition
- Flooring replaces values below a specified lower boundary with that boundary.
- It prevents unusually small values from falling below an acceptable limit.
- Flooring is useful when the lower extreme is problematic but the observations should be retained.

#### Business example
A business model does not allow negative inventory quantities.

If an analytical dataset contains negative quantities because of a measurement or processing issue, a lower limit may be applied after investigating the cause.
#### AI/ML use case
Flooring may be useful for:
- Numerical feature preprocessing.
- Measurement datasets.
- Some financial features.
- Variables with meaningful non-negative limits.

In [9]:
import pandas as pd

df = pd.DataFrame({
    "Quantity": [-10, -5, 0, 10, 20]
})

# Apply flooring
df["Quantity_Floored"] = df["Quantity"].clip(
    lower=0
)

print(df)

   Quantity  Quantity_Floored
0       -10                 0
1        -5                 0
2         0                 0
3        10                10
4        20                20


#### Explanation
- clip(lower=0) sets a minimum value of zero.
- Negative values become zero.
- Existing values above zero remain unchanged.

## Transformation
Definition
- Transformation changes the scale or mathematical representation of a variable.
- It can reduce skewness and make numerical distributions easier to model.
- Unlike capping, transformation generally preserves the ordering of values and does not necessarily replace extreme observations with a fixed boundary.
#### Real-world example
Income data may contain many small values and a few very large values.

Applying a logarithmic transformation can reduce the difference between extreme and ordinary values.
#### Business example
A company analyzes customer spending.

Most customers spend ₹500–₹2,000, while a few spend ₹1,00,000.
A logarithmic transformation may make the spending distribution easier to analyze.
#### AI/ML use case
Transformation is useful for:
- Linear Regression.
- Statistical modeling.
- Skewed numerical features.
- Reducing the influence of large magnitudes.
- Improving interpretability of relationships in some models.

In [10]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Income": [10000, 20000, 30000, 50000, 1000000]
})

# Log transformation
df["Income_Log"] = np.log1p(df["Income"])

print(df)

    Income  Income_Log
0    10000    9.210440
1    20000    9.903538
2    30000   10.308986
3    50000   10.819798
4  1000000   13.815512


#### Explanation
- np.log1p() calculates log(1 + x).
- It supports zero values safely.
- Large income values become numerically less extreme.
- The original Income column remains unchanged.

## Removing Outliers
Definition
- Removing outliers means deleting observations identified as extreme or invalid.
- It reduces the dataset size.
- Removal should be based on evidence that the observation is incorrect, irrelevant, or unsuitable for the analysis.
#### Real-world example
A person's age is recorded as 250 years due to a data entry mistake.
If the correct age cannot be recovered, the record may be excluded from an age-specific analysis.
#### Business example
A sales dataset contains an order amount of ₹99,99,99,999 due to an extra zero.
If the original transaction amount can be verified, correct it.

If it cannot be verified and is clearly invalid, the record may be excluded from the relevant analysis.
#### AI/ML use case
Removing invalid outliers may improve:
- Regression performance.
- Statistical analysis.
- Feature distributions.
- Data quality.

In [11]:
import pandas as pd

df = pd.DataFrame({
    "Salary": [20000, 25000, 30000, 35000, 40000, 500000]
})

# Calculate IQR boundaries
Q1 = df["Salary"].quantile(0.25)
Q3 = df["Salary"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Remove observations outside boundaries
clean_df = df[
    df["Salary"].between(lower, upper)
]

print("Original Data:")
print(df)

print("\nAfter Removal:")
print(clean_df)

Original Data:
   Salary
0   20000
1   25000
2   30000
3   35000
4   40000
5  500000

After Removal:
   Salary
0   20000
1   25000
2   30000
3   35000
4   40000


#### Explanation
- We calculated the IQR boundaries.
- We retained only values within those boundaries.
- Observations outside the boundaries were removed.
- The number of rows decreases.

## Retaining Outliers
Definition
- Retaining outliers means keeping extreme observations in the dataset without modifying their values.
- This is appropriate when unusual observations are valid and meaningful.
- Retaining outliers preserves the original information and avoids unnecessary data loss.

#### Business example
A bank observes a customer making a transaction worth ₹10,00,000.

The transaction is unusual but legitimate.
Removing it would incorrectly reduce the customer's transaction history.
#### AI/ML use case
Retaining outliers is especially important for:
- Fraud detection.
- Credit risk prediction.
- Rare event classification.
- Medical diagnosis.
- High-value customer identification.

In [12]:
import pandas as pd

df = pd.DataFrame({
    "Salary": [25000, 30000, 35000, 40000, 500000]
})

# Detect outliers using IQR
Q1 = df["Salary"].quantile(0.25)
Q3 = df["Salary"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df["Outlier"] = (
    (df["Salary"] < lower) |
    (df["Salary"] > upper)
)

# Retain all observations
print(df)

   Salary  Outlier
0   25000    False
1   30000    False
2   35000    False
3   40000    False
4  500000     True


#### Explanation
- We identified potential outliers using IQR.
- We created an Outlier column to flag unusual values.
- We did not delete or modify any observations.
- The original dataset remains intact.